In [12]:
!git clone https://github.com/Karthick-177/Flyrank-ML-Internship.git repo
%cd repo

Cloning into 'repo'...
remote: Enumerating objects: 134, done.
remote: Counting objects: 100% (134/134), done.
remote: Compressing objects: 100% (90/90), done.
remote: Total 134 (delta 46), reused 85 (delta 28), pack-reused 0 (from 0)
Receiving objects: 100% (134/134), 1.85 MiB | 9.54 MiB/s, done.
Resolving deltas: 100% (46/46), done.
/content/repo/repo


In [13]:
print(df.columns.tolist())

['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct', 'staleness_bucket', 'volume_bucket', 'reason_code', 'score']


# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Karthick-177/Flyrank-ML-Internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [14]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# Signal 1: Staleness (days_since_last_update)
df["staleness_bucket"] = pd.cut(df["days_since_last_update"], bins=[0, 90, 180, 10000], labels=["0-90d", "90-180d", "180d+"])
print(df.groupby("staleness_bucket").size())

# Signal 2: Volume (impressions_90d)
df["volume_bucket"] = pd.cut(df["impressions_90d"], bins=[0, 500, 2000, 1000000], labels=["low", "medium", "high"])
print(df.groupby("volume_bucket").size())
# Check staleness against trend direction
print(df.groupby("staleness_bucket")["trend_direction"].value_counts(normalize=True))

# Check volume against trend direction
print(df.groupby("volume_bucket")["trend_direction"].value_counts(normalize=True))

staleness_bucket
0-90d      20655
90-180d     9171
180d+        174
dtype: int64
volume_bucket
low       13285
medium     6502
high      10213
dtype: int64
staleness_bucket  trend_direction
0-90d             down               0.512031
                  stable             0.185863
                  up                 0.155217
                  new                0.103365
                  flat               0.043525
90-180d           down               0.611057
                  stable             0.228874
                  up                 0.125940
                  flat               0.025842
                  new                0.008287
180d+             down               0.471264
                  up                 0.155172
                  new                0.143678
                  stable             0.137931
                  flat               0.091954
Name: proportion, dtype: float64
volume_bucket  trend_direction
low            down               0.474671
             

/tmp/ipykernel_4468/2161267672.py:9: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  print(df.groupby("staleness_bucket").size())
/tmp/ipykernel_4468/2161267672.py:13: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  print(df.groupby("volume_bucket").size())
/tmp/ipykernel_4468/2161267672.py:15: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  print(df.groupby("staleness_bucket")["trend_direction"].value_counts(normalize=True))


Signal 1 — Staleness (days_since_last_update): MIXED. Decline rate rises from 51.2% (0-90d) to 61.1% (90-180d), but drops to 47.1% at 180d+ (n=174, small sample — less reliable). Not a clean monotonic relationship.

Signal 2 — Volume (impressions_90d): MIXED. Decline rate rises from 47.5% (low) to 61.8% (medium), but dips slightly to 58.1% (high). Medium-volume content declines most, not high or low.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [15]:
# Rule (plain words):
# A page is worth flagging for review if it's stale (90+ days since last update)
# and still has decent visibility (medium or high impressions).
# Stale-but-visible pages are the ones most worth refreshing.

# Reason codes:
# stale_but_visible - 90+ days old, medium/high impressions
# very_stale_but_visible - 180+ days old, medium/high impressions
# fresh_low_priority - recently updated, doesn't need review yet
# stale_but_invisible - old but low impressions, low priority despite age

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [16]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Encode the rule as a transparent score
def get_reason_code(row):
    if row["days_since_last_update"] >= 180 and row["impressions_90d"] >= 500:
        return "very_stale_but_visible"
    elif row["days_since_last_update"] >= 90 and row["impressions_90d"] >= 500:
        return "stale_but_visible"
    elif row["days_since_last_update"] < 90:
        return "fresh_low_priority"
    else:
        return "stale_but_invisible"

df["reason_code"] = df.apply(get_reason_code, axis=1)

# Score: readable on purpose, no fitted weights
stale = (df["days_since_last_update"] >= 90).astype(int)
visible = (df["impressions_90d"] >= 500).astype(int)
df["score"] = stale * visible * df["impressions_90d"]

# Rank and save
ranked = df.sort_values("score", ascending=False)
ranked_output = ranked[["content_id", "client_id", "score", "reason_code",
                          "days_since_last_update", "impressions_90d", "trend_direction"]]

import os
os.makedirs("work/outputs", exist_ok=True)
ranked_output.to_csv("work/outputs/baseline_action_score.csv", index=False)

print(ranked_output.head(10))

                 content_id          client_id   score        reason_code  \
6653   content_5fe46e04994d  client_4e07408562  517715  stale_but_visible   
29400  content_2dba2b1f9536  client_6208ef0f77  443434  stale_but_visible   
13537  content_2c2606c5d176  client_19581e27de  347399  stale_but_visible   
26531  content_cb112fce36be  client_19581e27de  309910  stale_but_visible   
21565  content_9532f197bbc8  client_4e07408562  309192  stale_but_visible   
3394   content_36ff89c8214e  client_19581e27de  295097  stale_but_visible   
26798  content_b28d1efd668f  client_6208ef0f77  286608  stale_but_visible   
23767  content_813e88069237  client_6208ef0f77  233561  stale_but_visible   
26255  content_c21024970297  client_19581e27de  211366  stale_but_visible   
7445   content_c8e9d6ab9013  client_19581e27de  208678  stale_but_visible   

       days_since_last_update  impressions_90d trend_direction  
6653                      104           517715            down  
29400                 

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [17]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
top20 = ranked_output.head(20)
print(top20[["content_id", "score", "reason_code", "days_since_last_update", "impressions_90d", "trend_direction"]].to_string())


                 content_id   score        reason_code  days_since_last_update  impressions_90d trend_direction
6653   content_5fe46e04994d  517715  stale_but_visible                     104           517715            down
29400  content_2dba2b1f9536  443434  stale_but_visible                     104           443434          stable
13537  content_2c2606c5d176  347399  stale_but_visible                     104           347399            down
26531  content_cb112fce36be  309910  stale_but_visible                     104           309910            down
21565  content_9532f197bbc8  309192  stale_but_visible                     104           309192            down
3394   content_36ff89c8214e  295097  stale_but_visible                     104           295097          stable
26798  content_b28d1efd668f  286608  stale_but_visible                     104           286608          stable
23767  content_813e88069237  233561  stale_but_visible                     104           233561         

content_5fe46e04994d — Review for refresh. Reason: stale_but_visible (104d, 517,715 impr). Confidence: medium. Wrong if: trend is "down" already reflects a real decline unrelated to staleness (e.g. seasonal drop) — refresh may not fix it.
content_2dba2b1f9536 — Review for refresh. Reason: stale_but_visible (104d, 443,434 impr). Confidence: low. Wrong if: trend is "stable" — page isn't actually struggling, refresh priority should be lower.
content_2c2606c5d176 — Review for refresh. Reason: stale_but_visible (104d, 347,399 impr). Confidence: high. Wrong if: the "down" trend is driven by a market/seasonal shift, not staleness.
content_cb112fce36be — Review for refresh. Reason: stale_but_visible (104d, 309,910 impr). Confidence: high. Wrong if: decline is due to a competitor change, not content age.
content_9532f197bbc8 — Review for refresh. Reason: stale_but_visible (104d, 309,192 impr). Confidence: high. Wrong if: same as above — decline unrelated to staleness.
content_36ff89c8214e — Review for refresh. Reason: stale_but_visible (104d, 295,097 impr). Confidence: low. Wrong if: "stable" trend means it doesn't need refreshing yet.
content_b28d1efd668f — Review for refresh. Reason: stale_but_visible (104d, 286,608 impr). Confidence: low. Wrong if: stable trend suggests refresh isn't urgent.
content_813e88069237 — Review for refresh. Reason: stale_but_visible (104d, 233,561 impr). Confidence: high. Wrong if: decline is unrelated to content age.
content_c21024970297 — Review for refresh. Reason: stale_but_visible (104d, 211,366 impr). Confidence: low. Wrong if: stable trend means low urgency.
content_c8e9d6ab9013 — Review for refresh. Reason: stale_but_visible (104d, 208,678 impr). Confidence: high. Wrong if: decline driven by external factors, not staleness.
content_b511d4bc4ad2 — Review for refresh. Reason: stale_but_visible (104d, 205,915 impr). Confidence: low. Wrong if: stable trend, low urgency.
content_d17681677e69 — Review for refresh. Reason: stale_but_visible (104d, 201,584 impr). Confidence: low. Wrong if: stable trend, low urgency.
content_a7427266c305 — Review for refresh. Reason: stale_but_visible (104d, 201,111 impr). Confidence: low. Wrong if: stable trend, low urgency.
content_c5063073d048 — Review for refresh. Reason: stale_but_visible (104d, 192,205 impr). Confidence: low. Wrong if: stable trend, low urgency.
content_3d94572c3a35 — Review for refresh. Reason: stale_but_visible (104d, 190,623 impr). Confidence: high. Wrong if: decline unrelated to staleness.
content_01908772c6db — Review for refresh. Reason: stale_but_visible (104d, 187,893 impr). Confidence: high. Wrong if: decline unrelated to staleness.
content_33b4dceecad1 — Review for refresh. Reason: stale_but_visible (104d, 181,574 impr). Confidence: low. Wrong if: stable trend, low urgency.
content_f02b48f88241 — Weak pick. Reason: stale_but_visible (104d, 181,514 impr), but trend is already "up." Confidence: very low — this page is improving without a refresh, so flagging it is likely wasted effort.
content_05e9b4cd9ccf — Review for refresh. Reason: stale_but_visible (104d, 179,002 impr). Confidence: high. Wrong if: decline unrelated to staleness.
content_bb5bd5f771dc — Review for refresh. Reason: stale_but_visible (104d, 176,296 impr). Confidence: low. Wrong if: stable trend, low urgency.

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

# Weak pick: content_f02b48f88241 (rank 18)
# Flagged as stale_but_visible, but trend_direction is "up" — already improving.
# The rule doesn't check trend direction, so it can't distinguish "stale and declining"
# from "stale but already recovering." This is a logic gap, not a data error.
#
# Leakage check: No labels (trend_direction, trend_pct) or future-window columns were
# used as INPUTS to the score — only days_since_last_update and impressions_90d, both
# available at scoring time. trend_direction was used only for review/evaluation, not
# as a feature. No leakage.

In [18]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Weak pick: content_f02b48f88241 (rank 18)
# Flagged as stale_but_visible, but trend_direction is "up" — already improving.
# The rule doesn't check trend direction, so it can't distinguish "stale and declining"
# from "stale but already recovering." This is a logic gap, not a data error.
#
# Leakage check: No labels (trend_direction, trend_pct) or future-window columns were
# used as INPUTS to the score — only days_since_last_update and impressions_90d, both
# available at scoring time. trend_direction was used only for review/evaluation, not
# as a feature. No leakage.

Self-check:
- The rule fits in three sentences a non-engineer could understand: yes (stale + visible = worth reviewing).
- No labels or future-window columns (trend_direction, trend_pct) were used as inputs to the score —
  only days_since_last_update and impressions_90d, both available at scoring time.
- The top-20 hand review found at least one weak pick (content_f02b48f88241, rank 18, flagged
  despite an "up" trend) — this satisfies the "found at least one weak pick" requirement.
- Small-sample caution: the 180d+ staleness bucket only had 174 rows, so conclusions there
  are less reliable than the 0-90d and 90-180d buckets.
- CSV was written to work/outputs/baseline_action_score.csv and regenerates on every run.